# 02 — Data Quality Analysis

**Goal:** Understand what good vs. bad training examples look like before we clean anything.

**Day 13** of the implementation plan.

In [ ]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset('lavita/ChatDoctor-HealthCareMagic-100k')
df = pd.DataFrame(dataset['train'])
print(f'Total examples: {len(df)}')

In [ ]:
# --- Answer length distribution ---
df['answer_length'] = df['output'].str.len()
df['question_length'] = df['input'].str.len()

print('=== Answer (output) length stats ===')
print(df['answer_length'].describe())

print('\n=== Question (input) length stats ===')
print(df['question_length'].describe())

In [ ]:
# --- Identify bad examples ---

# Too short to be useful
short_answers = df[df['answer_length'] < 100]
print(f'Short answers (< 100 chars): {len(short_answers)} ({len(short_answers)/len(df)*100:.1f}%)')
print('\nSamples of short answers:')
print(short_answers['output'].head(5).tolist())

# Too long (exceeds model context window)
long_answers = df[df['answer_length'] > 2000]
print(f'\nLong answers (> 2000 chars): {len(long_answers)} ({len(long_answers)/len(df)*100:.1f}%)')

# Empty questions
empty_questions = df[df['question_length'] < 10]
print(f'\nEmpty questions (< 10 chars): {len(empty_questions)} ({len(empty_questions)/len(df)*100:.1f}%)')

In [ ]:
# --- Manually review 20 random examples ---
print('=== 20 Random Examples for Manual Review ===\n')
random_sample = df.sample(20, random_state=7)

for i, (_, row) in enumerate(random_sample.iterrows(), 1):
    print(f'[{i}] Q: {row["input"][:200]}')
    print(f'    A: {row["output"][:300]}')
    print()

In [ ]:
# --- Summary ---
keep_count = len(df[(df['answer_length'] >= 100) & 
                    (df['answer_length'] <= 2000) & 
                    (df['question_length'] >= 10)])

print(f'Examples to keep after cleaning: {keep_count} ({keep_count/len(df)*100:.1f}%)')
print(f'Examples to remove: {len(df) - keep_count}')

# After running this, write in LEARNINGS.md:
# - What makes a good example vs a bad one?
# - What filtering thresholds make sense?